# Uncertainty Analysis Tutorial
## Two-Compartment Antibody PK Model

**Certara IQ Python Client**  
*Quantifying parameter uncertainty in a monoclonal antibody pharmacokinetic model*

This notebook walks through the complete uncertainty quantification workflow in IQ Analyze using a two-compartment PK model for a 150 kDa monoclonal antibody.

## Overview

| Step | Function | Purpose |
|------|----------|---------|
| 1 | `simulate()` | Baseline simulation with nominal parameters |
| 2 | `optimize()` | Fit PK parameters to observed concentration data |
| 3 | `confidence_intervals()` | Fast frequentist CIs via Fisher Information Matrix |
| 3.1 | `simulate()` | Prediction intervals via Laplace parameter sampling |
| 4 | `profile_likelihood()` | Accurate frequentist CIs with identifiability diagnostics |
| 5 | `parameter_posterior_sample()` | Full Bayesian posterior via MCMC |
| 5.1 | `simulate()` | Prediction intervals from posterior draws |

### The Model

The model file `two_compartment_pk_antibody.txt` (V2 reaction model format) describes:

- **Central compartment** (plasma, Vc = 3 L): drug is dosed and eliminated
- **Peripheral compartment** (tissue, Vp = 4 L): drug distributes reversibly via Q

```
central  <-[k12/k21]->  peripheral
   |
 [kel]
  sink
```

Key parameters: CL (systemic clearance), Q (inter-compartmental clearance), Vc, Vp.

The model exposes a dosing route: `IV_mg` (intravenous bolus, mg).

Outputs: `C_central_nM` (plasma concentration, nM) and `C_peripheral_nM` (tissue concentration, nM).


In [ ]:
import abm
import pandas as pd
import numpy as np
import math
import plotnine as p9
from utilities.uncertainty import covariance_matrix
import seaborn as sns
import matplotlib.pyplot as plt

MODEL = "inputs/uncertainty_analysis/two_compartment_pk_antibody.txt"

---
## 1. Baseline Simulation

We verify the model by running a single 10 mg IV bolus simulation with the nominal parameters before fitting.

In [ ]:
# Single parameter table used for both simulation and optimization.
# The extra columns (is_fit, bounds, priors) are ignored by simulate().
parameters = pd.DataFrame({
    'parameter':          ['CL',         'Q',          'Vc',         'Vp'],
    'value':              [0.012,        0.016,        3.0,          4.0],
    'unit':               ['L/hr',       'L/hr',       'L',          'L'],
    'is_fit':             [True,         True,        True,         False],
    'lower_bound':        [0.001,        0.001,        0.5,          math.nan],
    'upper_bound':        [1,            1,            15.0,         math.nan],
    'prior_distribution': ['loguniform', 'loguniform', 'loguniform', 'loguniform'],
    'location':           [math.nan,     math.nan,     math.nan,     math.nan],
    'scale':              [math.nan,     math.nan,     math.nan,     math.nan],
})

# 10 mg IV bolus at time 0
doses_iv = pd.DataFrame({
    'route':       ['IV_mg'],
    'times':       [0.0],
    'time_unit':   ['hr'],
    'amounts':     [10.0],
    'amount_unit': ['mg'],
})

In [ ]:
# 200 linearly spaced time points from 0 to 504 hr (3 weeks)
times = abm.linspace(0, 504, 200, "hr")

sim_result = abm.simulate(
    models=MODEL,
    parameters=parameters,
    doses=doses_iv,
    times=times,
    outputs=["C_central_nM", "C_peripheral_nM"]
)

df = sim_result.to_pandas(tall_outputs=True)
print(df.head())

In [ ]:
(
    p9.ggplot(df.query('value > 0.'), p9.aes(x="t", y="value", color="output"))
    + p9.geom_line(size=1)
    + p9.scale_y_log10()
    + p9.scale_color_manual(values=["steelblue", "tomato"], 
                            labels={'C_central_nM': 'central', 'C_peripheral_nM': 'peripheral'})
    + p9.labs(
        x="Time (hr)",
        y="Concentration (nM)",
        color="",
        title="Two-compartment antibody PK — 10 mg IV bolus",
    )
    + p9.theme_bw()
    + p9.theme(figure_size=(8, 4))
)

---
## 2. Parameter Optimization

`optimize()` fits CL, Q, and Vc to observed plasma concentration data using the fides-BFGS algorithm. The synthetic observations below represent a Phase 1 single-dose PK study (10 mg IV bolus, serial sampling over 3 weeks).

Measurement error is modeled with **15% proportional error**.

In [ ]:
# Synthetic observed C_central_nM — 10 mg IV bolus, representative Phase 1 patient
# Values generated from the nominal model with ~15% proportional measurement noise
measurements = pd.DataFrame({
    'time':               [1,    4,    8,    24,   48,   96,    168,  336,  504 ],
    'time_unit':          ['hr'] * 9,
    'output':             ['C_central_nM'] * 9,
    'measurement':        [21.3, 20.8, 21.5, 18.9, 13.8, 11.1,  6.3,  4.2,  2.7],
    'measurement_unit':   ['nM'] * 9,
    'constant_error':     [0.0]  * 9,
    'proportional_error': [0.15] * 9,   # 15% proportional error
    'exponential_error':  [0.0]  * 9,
})

In [ ]:
opt_result = abm.optimize(
    measurements=measurements,
    models=MODEL,
    parameters=parameters,
    doses=doses_iv
)

# Fitted parameter values with units
print("Fitted parameters:")
display(opt_result.fit_parameter_table.to_pandas())

In [ ]:
# Simulate with the fitted parameters over a dense time grid
fitted_sim = opt_result.simulate(
    times=abm.linspace(0, 504, 200, "hr"),
    outputs=["C_central_nM", "C_peripheral_nM"],
)
fitted_df = fitted_sim.to_pandas(tall_outputs=False)
print("Fitted simulation (first 5 rows):")
display(fitted_df.head())

In [ ]:
(
    p9.ggplot()
    + p9.geom_line(
        data=fitted_df,
        mapping=p9.aes(x="t", y="C_central_nM"),
        size=1,
    )
    + p9.geom_point(
        data=measurements,
        mapping=p9.aes(x="time", y="measurement"),
        color="steelblue",
        size=3,
    )
    + p9.scale_y_log10()
    + p9.labs(
        x="Time (hr)",
        y="C_central (nM)",
        title="Goodness-of-fit — observed (points) vs. fitted (line)",
    )
    + p9.theme_bw()
    + p9.theme(figure_size=(8, 4))
)

---
## 3. Uncertainty Quantification

After optimization, three complementary methods quantify uncertainty in the fitted parameters. Each makes different assumptions and offers different trade-offs.

| Method | Framework | Key assumption | Speed |
|--------|-----------|----------------|-------|
| `confidence_intervals()` | Frequentist (FIM / Laplace) | Gaussian curvature at optimum | Fast |
| `profile_likelihood()` | Frequentist | None (also diagnoses identifiability) | Moderate |
| `parameter_posterior_sample()` | Bayesian MCMC | Prior from parameters table | Slow |

Use `confidence_intervals()` for a fast first look. Use `profile_likelihood()` to check identifiability or when Gaussian assumptions may not hold. Use `parameter_posterior_sample()` when the full posterior is needed for downstream propagation.

### 3.1 Confidence Intervals — Fisher Information Matrix (Laplace Approximation)

`confidence_intervals()` computes frequentist confidence intervals using the curvature of the objective at the optimum. This is the fastest method but assumes the posterior is approximately Gaussian near the solution.

`confidence_intervals()` returns a table with the following columns:

| Column      | Type               | Description                                                                                                                                |
|-------------|--------------------|--------------------------------------------------------------------------------------------------------------------------------------------|
| `parameter` | `str`              | Name of the fitted global parameter.                                                                                                       |
| `value`     | `float`            | Maximum-likelihood (point-estimate) value of the parameter, in its natural scale.                                                          |
| `unit`      | `str`              | Physical unit of the parameter (e.g. `"1/day"`), sourced from the fitted optimization output.                                              |
| `scale`     | `"linear" \| "log"` | Whether the parameter was estimated on a linear or log scale, determined by its prior.                                                    |
| `lower`     | `float`            | Lower bound of the confidence interval, back-transformed to the natural (unscaled) space.                                                  |
| `upper`     | `float`            | Upper bound of the confidence interval, back-transformed to the natural (unscaled) space.                                                  |

In [ ]:
# 95% CIs via FIM (fast; assumes Gaussian curvature at optimum)
ci_95 = opt_result.confidence_intervals(fraction=0.95)
print("95% confidence intervals (FIM):")
display(ci_95)

In [ ]:
(
    p9.ggplot(ci_95, p9.aes(x="value", y="parameter", xmin="lower", xmax="upper"))
    + p9.geom_errorbarh(height=0.3)
    + p9.geom_point(size=3, color="steelblue")
    + p9.scale_x_log10()
    + p9.labs(
        x="Parameter value (log scale)",
        y="",
        title="95% confidence intervals — Fisher Information Matrix",
    )
    + p9.theme_bw()
    + p9.theme(figure_size=(7, 3))
)

#### 3.1.1 Prediction Intervals via Laplace Sampling

`confidence_intervals()` reports intervals on **parameters**. To propagate that uncertainty into model **predictions**, we sample parameter sets from the approximate multivariate normal distribution defined by the FIM covariance matrix, simulate each sample, and compute empirical percentiles across the resulting trajectories.

Because the fitted parameters use `loguniform` priors, the optimizer works in log-space and the FIM covariance is with respect to log-parameters. We therefore sample in log-space and exponentiate before simulating.

#### `covariance_matrix(opt_result)`

`covariance_matrix` returns the **parameter covariance matrix** derived from the Fisher Information Matrix (FIM) evaluated at the MAP (maximum a posteriori) estimate.

The FIM measures how sharply the objective function is curved around the optimum — a steep, narrow valley means the data strongly constrain a parameter, while a shallow bowl means it is weakly identified. Inverting the FIM converts that curvature into a covariance matrix: diagonal entries are the marginal variances of each fitted parameter, and off-diagonal entries capture how parameters co-vary.

Because the parameters in this model use `loguniform` priors the optimizer internally works in **log-space**, so the FIM and its inverse are also expressed in log-space. Sampling from a multivariate normal defined by this matrix (and then exponentiating) produces draws that respect both the parameter scale and the correlations the optimizer discovered.

| Argument | Default | Effect |
|---|---|---|
| `opt_result` | — | `OptimizationResult` returned by `optimize()` |
| `uncertainty_ceiling` | `inf` | Caps the standard deviation (sqrt of each diagonal entry) before returning. Useful when a parameter is near-non-identifiable and FIM inversion is numerically unstable; a value of `1e8` is a reasonable guard. |

**Returns:** `np.ndarray` of shape `(n_fit_params, n_fit_params)` — row/column order matches the fitted parameters in `opt_result.fit_parameter_table`.

In [ ]:
N_LAPLACE = 200  # parameter samples to draw from the Laplace approximation

fit_table = opt_result.fit_parameter_table.to_pandas()

# Sample in log-space from the multivariate normal defined by the FIM
log_means = np.log(fit_table.query('is_fit').value.values.astype(float))
cov_matrix = covariance_matrix(opt_result) # log-space covariance matrix

rng = np.random.default_rng(seed=42)
log_draws = rng.multivariate_normal(log_means, cov_matrix, size=N_LAPLACE)
nat_draws = np.exp(log_draws)   # shape (N_LAPLACE, n_fit_params) — natural scale

# Non-fitted parameters — matched to every simulation via sample_id = "*"
non_fitted = (
    fit_table.query('not is_fit')
    [["parameter", "value", "unit"]]
    .assign(sample_id="*")
)

# Fitted parameters — one row per parameter per sampled draw
sample_rows = [
    {"parameter": pname, "value": float(nat_draws[i, j]),
     "unit": punit, "sample_id": str(i)}
    for i in range(N_LAPLACE)
    for j, (pname, punit) in enumerate(fit_table.query('is_fit')[['parameter', 'unit']].values)
]

laplace_params = pd.concat([non_fitted, pd.DataFrame(sample_rows)], ignore_index=True)
simulations_laplace = pd.DataFrame({"sample_id": [str(i) for i in range(N_LAPLACE)]})

laplace_pred = abm.simulate(
    simulations=simulations_laplace,
    models=MODEL,
    parameters=laplace_params,
    doses=doses_iv,
    times=abm.linspace(0, 504, 100, "hr"),
    outputs=["C_central_nM"]
)
print(f"Simulated {N_LAPLACE} Laplace samples.")

In [ ]:
laplace_sim_df = laplace_pred.to_pandas(tall_outputs=False)

# 2.5th / 50th / 97.5th percentiles across samples at each time point
pi_laplace = (
    laplace_sim_df.groupby("t")["C_central_nM"]
    .quantile([0.025, 0.5, 0.975])
    .unstack()
    .reset_index()
    .rename(columns={0.025: "lo", 0.5: "mid", 0.975: "hi"})
)

(
    p9.ggplot(pi_laplace, p9.aes(x="t"))
    + p9.geom_ribbon(p9.aes(ymin="lo", ymax="hi"), alpha=0.25, fill="steelblue")
    + p9.geom_line(p9.aes(y="mid"), color="steelblue", size=1)
    + p9.geom_line(
        data=fitted_df,
        mapping=p9.aes(x="t", y="C_central_nM"),
        color="black", linetype="dashed", size=0.8,
    )
    + p9.geom_point(
        data=measurements,
        mapping=p9.aes(x="time", y="measurement"),
        color="black", size=2.5,
    )
    + p9.scale_y_log10()
    + p9.labs(
        x="Time (hr)",
        y="C_central (nM)",
        title=f"95% prediction interval — Laplace approximation ({N_LAPLACE} samples)",
    )
    + p9.theme_bw()
    + p9.theme(figure_size=(8, 4))
)

### 3.2 Profile Likelihood

`profile_likelihood()` traces the objective function as each parameter is stepped away from its optimum, re-optimizing all other parameters at each step. Confidence interval bounds are where the profile crosses the chi-squared threshold.

The returned `DataPipeResult` has one row per (parameter, direction, profile step). The `termination` column indicates why the profile stopped:

| `termination` | Meaning |
|---------------|---------|
| `"threshold"` | CI bound found (profile crossed chi-squared threshold) |
| `"bound"` | Parameter bound reached — possibly unidentifiable |
| `"max_iterations"` | Step limit reached |
| `"reversal"` | Descent in objective detected (possible secondary minimum) |
| `"error"` | Solver error |

In [ ]:
profile = opt_result.profile_likelihood(
    fraction=0.95,
    simultaneous_intervals=True,          # Correct for multiple comparisons (df = n_fit_params)
    precision=0.1,                         # Smaller → finer profile curves
    max_iterations=None,                   # Default: 10 / precision = 100 steps per direction
    min_step=1e-6,
    max_step=0.18,
    reversal_abstol=1,
    reversal_reltol=0.1,
    threshold_method="linear_interpolation",  # More precise than "step_over"
)

profile_df = profile.to_pandas()
print("Profile likelihood result (first 5 rows):")
display(profile_df.head(5))

# Identifiability summary — check that all profiles terminate at 'threshold'
term_summary = (
    profile_df[["parameter_name", "direction", "termination", "parameter_threshold"]]
    .drop_duplicates()
    .sort_values(["parameter_name", "direction"])
    .reset_index(drop=True)
)
print("\nProfile termination summary:")
display(term_summary)

In [ ]:
# The chi-squared threshold is constant across all parameters
threshold_val = profile_df["objective_threshold"].dropna().iloc[0]

# One CI bound row per (parameter, direction) for vertical reference lines
threshold_df = (
    profile_df[["parameter_name", "direction", "parameter_threshold"]]
    .drop_duplicates()
    .dropna(subset=["parameter_threshold"])
)

(
    p9.ggplot(profile_df, p9.aes(x="parameter_value", y="objective_value",
                                  color="direction"))
    + p9.geom_line(size=0.8)
    + p9.geom_hline(yintercept=threshold_val, linetype="dashed", color="black")
    + p9.geom_vline(
        data=threshold_df,
        mapping=p9.aes(xintercept="parameter_threshold"),
        linetype="dashed",
        color="black",
        inherit_aes=False,
    )
    + p9.facet_wrap("parameter_name", scales="free_x", ncol=2)
    + p9.labs(
        x="Parameter value",
        y="Objective G(θ)",
        color="Direction",
        title="Profile likelihood — 95% simultaneous confidence intervals",
    )
    + p9.theme_bw()
    + p9.theme(figure_size=(8, 6))
)

### 3.3 Bayesian Posterior via MCMC

`parameter_posterior_sample()` draws samples from the Bayesian posterior $p(\theta \mid \text{data})$ using Markov Chain Monte Carlo. The posterior is proportional to the likelihood (evaluated by integrating the ODE) times the prior distributions from the parameters table.

Each entry in `seeds` launches an independent chain starting from the optimizer result. Running at least 4 chains enables convergence diagnostics:

| Column       | Type    | Description                                                                                     | Healthy value      |
|--------------|---------|-------------------------------------------------------------------------------------------------|--------------------|
| `parameter`  | str     | Name of the fitted parameter                                                                    | —                  |
| `mean`       | float   | Posterior mean across all chains and draws                                                      | —                  |
| `sd`         | float   | Posterior standard deviation across all chains and draws                                        | —                  |
| `hdi_3%`     | float   | Lower bound of the 94% highest-density interval (HDI)                                          | —                  |
| `hdi_97%`    | float   | Upper bound of the 94% highest-density interval (HDI)                                          | —                  |
| `mcse_mean`  | float   | Monte Carlo standard error on the mean; uncertainty in the mean estimate due to finite sampling | Small relative to `sd` |
| `mcse_sd`    | float   | Monte Carlo standard error on the standard deviation estimate                                   | Small relative to `sd` |
| `ess_bulk`   | float   | Effective sample size for the bulk of the distribution; accounts for autocorrelation            | ≥ 100 per chain    |
| `ess_tail`   | float   | Effective sample size for the tails; important for credible interval accuracy                   | ≥ 100 per chain    |
| `r_hat`      | float   | Gelman-Rubin statistic; ratio of between-chain to within-chain variance                        | ≤ 1.01             |

In [ ]:
%%time

# NUTS — No-U-Turn Sampler (default; recommended for smooth posteriors)
posterior = opt_result.parameter_posterior_sample(
    n=1000,              # Posterior draws per chain after tuning
    seeds=[1, 2, 3, 4],  # 4 independent chains
    method="NUTS",
    tune=1000,           # Warm-up steps for step-size and mass-matrix adaptation
    discard_tuned_samples=True,
    thin=1,              # Keep every draw; increase if autocorrelation is high
)

# Posterior samples — one row per draw, columns are fitted parameter names
samples_df = posterior.samples.to_pandas()
print(f"Posterior samples shape: {samples_df.shape}  (n_chains × n_draws = 4 × 1000)")
print(samples_df.head())

# Convergence diagnostics
print(f"\nDivergences per chain: {posterior.n_divergences}")
print("\nConvergence diagnostics:")
posterior.diagnostics.to_pandas()

In [ ]:
g = sns.PairGrid(samples_df, diag_sharey=False)
g.map_diag(sns.histplot, bins=40, color="steelblue", alpha=0.8)
g.map_offdiag(sns.scatterplot, alpha=0.3, s=4, color="steelblue")
g.figure.suptitle(
    "Posterior joint distributions — NUTS (4 chains × 1000 draws)",
    y=1.02,
)
g.figure.set_dpi(100)
g.figure.set_size_inches(6, 6)
plt.tight_layout()
plt.show()

#### 3.3.1 Prediction Intervals from Posterior Samples

The NUTS posterior samples encode the full joint distribution over parameters. We pass each draw through the model using `simulate()` and aggregate the resulting trajectories into empirical prediction intervals. This approach captures all posterior uncertainty including parameter correlations — making it the most principled way to propagate fitted uncertainty into forward predictions.

To keep compute time manageable we thin the 4 × 1000 posterior draws down to a working set before simulating.

In [ ]:
N_POST = 1000

# Thin samples and convert to tall format
thin_step = max(1, len(samples_df) // N_POST)
sampled_parameters = (
    samples_df.iloc[::thin_step].iloc[:N_POST]
    .reset_index(drop=True)
    .assign(sample_id=lambda df: pd.Series(range(df.shape[0])).astype('str'))
    .melt(id_vars='sample_id', var_name='global_parameter_name', value_name='value')
)

# Add samples to original fitted parameter table and drop unneeded columns
post_params = (
    opt_result.fit_parameter_table.to_pandas()
    .rename(columns={'value': 'value_0'})
    .merge(sampled_parameters, how='left', on='global_parameter_name')
    .assign(
        value=lambda df: df.value.mask(df.value.isna(), df.value_0),
        sample_id=lambda df: df.sample_id.fillna('*')
    )
    .drop(columns=['value_0', 'is_fit', 'lower_bound', 'upper_bound',
                   'prior_distribution', 'location',
                  'scale', 'global_parameter_name'])
)

sample_ids = sampled_parameters.sample_id.unique()
simulations_post = pd.DataFrame({"sample_id": sample_ids})

post_pred = abm.simulate(
    simulations=simulations_post,
    models=MODEL,
    parameters=post_params,
    doses=doses_iv,
    times=abm.linspace(0, 504, 100, "hr"),
    outputs=["C_central_nM"]
)

print(f"Simulated {len(sample_ids)} posterior draws.")

In [ ]:
post_sim_df = post_pred.to_pandas(tall_outputs=False)

# 2.5th / 50th / 97.5th percentiles across posterior draws at each time point
pi_post = (
    post_sim_df.groupby("t")["C_central_nM"]
    .quantile([0.025, 0.5, 0.975])
    .unstack()
    .reset_index()
    .rename(columns={0.025: "lo", 0.5: "mid", 0.975: "hi"})
)

(
    p9.ggplot(pi_post, p9.aes(x="t"))
    + p9.geom_ribbon(p9.aes(ymin="lo", ymax="hi"), alpha=0.25, fill="darkorange")
    + p9.geom_line(p9.aes(y="mid"), color="darkorange", size=1)
    + p9.geom_line(
        data=fitted_df,
        mapping=p9.aes(x="t", y="C_central_nM"),
        color="black", linetype="dashed", size=0.8,
    )
    + p9.geom_point(
        data=measurements,
        mapping=p9.aes(x="time", y="measurement"),
        color="black", size=2.5,
    )
    + p9.scale_y_log10()
    + p9.labs(
        x="Time (hr)",
        y="C_central (nM)",
        title=f"95% prediction interval — NUTS posterior ({len(sample_ids)} draws)",
    )
    + p9.theme_bw()
    + p9.theme(figure_size=(8, 4))
)

#### Alternative MCMC samplers

**Hamiltonian MC** with fixed leapfrog steps.

In [ ]:
%%time

posterior_hmc = opt_result.parameter_posterior_sample(
    n=500,
    seeds=[10, 11, 12, 13],
    method="HamiltonianMC",
    tune=1000,
    discard_tuned_samples=True,
    thin=1,
)

**Metropolis** — gradient-free; useful when gradients are unavailable. Requires more thinning because random-walk chains mix slowly.

In [ ]:
%%time

posterior_metro = opt_result.parameter_posterior_sample(
    n=500,
    seeds=[20, 21, 22, 23],
    method="Metropolis",
    tune=2000,
    discard_tuned_samples=True,
    thin=5,
)

**DEMetropolisZ** — differential-evolution variant; useful for correlated posteriors.

In [ ]:
%%time

posterior_de = opt_result.parameter_posterior_sample(
    n=1000,
    seeds=[30, 31, 32, 33],
    method="DEMetropolisZ",
    tune=2000,
    discard_tuned_samples=True,
    thin=3,
)

**Slice sampler** — gradient-free alternative.

In [ ]:
%%time

posterior_slice = opt_result.parameter_posterior_sample(
    n=500,
    seeds=[40, 41, 42, 43],
    method="Slice",
    tune=500,
    discard_tuned_samples=True,
    thin=2,
)

---
## Summary

This tutorial demonstrated the IQ Python client uncertainty quantification workflow applied to a two-compartment antibody PK model.

### Functions and result types

| Function | Result type | Key outputs |
|----------|-------------|-------------|
| `simulate()` | `SimulationResult` | Time-course predictions |
| `optimize()` | `OptimizationResult` | `fit_parameter_table`; `simulate()`, `residuals()` |
| `confidence_intervals()` | `DataFrame` | Frequentist CIs via FIM (fast) |
| `profile_likelihood()` | `DataPipeResult` | Frequentist CIs with identifiability diagnostics |
| `parameter_posterior_sample()` | `PosteriorSampleResult` | `samples`, `diagnostics`, `n_divergences` |

### Prediction interval approach

Both the Laplace and posterior sampling methods can be used to propagate parameter uncertainty into model predictions:

| Approach | Source of parameter samples | How to propagate |
|----------|-----------------------------|------------------|
| Laplace | Draw from the multivariate normal defined by the FIM covariance (log-space) | Exponentiate draws, run `simulate()` on all samples |
| Posterior (MCMC) | Use `PosteriorSampleResult.samples` directly | Thin draws as needed, run `simulate()` on all draws |

In both cases the workflow is the same: 
1. Build a `simulations` table with a `sample_id` label column
2. Build a `parameters` table with one row per fitted parameter per sample and one row per fixed parameter (each fixed parameter uses `sample_id = "*"` to match all)
3. Call `abm.simulate()` with both tables
4. Compute percentile bands from the resulting trajectories.

### Uncertainty method guidance

- Use `confidence_intervals()` for a fast first look immediately after optimization.
- Use `profile_likelihood()` to check identifiability and when the Gaussian approximation may not hold (e.g., near parameter bounds or when profiles are asymmetric).
- Use `parameter_posterior_sample(method="NUTS")` when you need the full joint posterior — especially to propagate uncertainty into downstream model predictions.
- For prediction intervals, the posterior approach is the most principled; the Laplace approach is a fast approximation that works well when the posterior is close to Gaussian.
